### Similarity Search

In [108]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv

load_dotenv()

True

In [44]:
hf_embeddings = HuggingFaceEmbeddings(model="all-minilm-l6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3092.83it/s]


### Cosine Similarity

In [ ]:
def cosine_similarity(vect1, vect2):
    dot_prod = np.dot(vect1, vect2)
    norm_a = np.linalg.norm(vect1)
    norm_b = np.linalg.norm(vect2)
    return dot_prod / (norm_a * norm_b)

In [46]:
documents = [
    "Ya'Allah please help me to achieve my dream of going to graduate school in the USA",
    "I love USA and It's a place I want to study",
    "Let's write Python codes",
    "Langchain, Langgraph and RAG are interesting"
]

emd_docs = hf_embeddings.embed_documents(documents)
emd_docs

[[0.08453284949064255,
  0.049279049038887024,
  0.02707751654088497,
  0.03013215959072113,
  -0.021261097863316536,
  -0.008329873904585838,
  -0.03837208449840546,
  -0.08986906707286835,
  0.0015737480716779828,
  -0.005789016839116812,
  -0.03186550363898277,
  0.000909532536752522,
  0.013154208660125732,
  -0.030556639656424522,
  -0.00028216707869432867,
  0.050336986780166626,
  -0.04856891930103302,
  -0.049145158380270004,
  -0.00737717654556036,
  -0.11901894956827164,
  0.03448925167322159,
  0.05293576046824455,
  -0.0020035163033753633,
  -0.042652565985918045,
  0.052296508103609085,
  0.035897962749004364,
  0.038759104907512665,
  -0.026097603142261505,
  0.03375818580389023,
  -0.0029873100575059652,
  0.03051823377609253,
  -0.03447046875953674,
  -0.06522445380687714,
  0.014994239434599876,
  0.04870430752635002,
  0.06714481860399246,
  0.080641008913517,
  0.00627153180539608,
  0.058129094541072845,
  0.000653061899356544,
  0.06165586784482002,
  0.01591528765

In [47]:
query = "What is Python"
emd_query = hf_embeddings.embed_query(query)

cosine_similarity(emd_query, emd_docs[2])

np.float64(0.5491086047886008)

In [48]:
# Compare each document against all others to see their similarities
for i, doc in enumerate(documents):
    for j in range(len(documents)):
        # TODO: Comment out if you don't want to compare against itself
        # if doc == documents[j]:
        #     continue
        query = hf_embeddings.embed_query(doc)
        score = cosine_similarity(query, emd_docs[j])
        # print(f"{doc} ◀ VS ▶ {documents[j]} \n Score {score} \n")
        print(f"{doc} 🆚 {documents[j]} \n Score {score} \n")
    print("="*50)
        

Ya'Allah please help me to achieve my dream of going to graduate school in the USA 🆚 Ya'Allah please help me to achieve my dream of going to graduate school in the USA 
 Score 0.9999999999998884 

Ya'Allah please help me to achieve my dream of going to graduate school in the USA 🆚 I love USA and It's a place I want to study 
 Score 0.5477496704972038 

Ya'Allah please help me to achieve my dream of going to graduate school in the USA 🆚 Let's write Python codes 
 Score 0.0446280409050498 

Ya'Allah please help me to achieve my dream of going to graduate school in the USA 🆚 Langchain, Langgraph and RAG are interesting 
 Score -0.018254120665266438 

I love USA and It's a place I want to study 🆚 Ya'Allah please help me to achieve my dream of going to graduate school in the USA 
 Score 0.5477497414004566 

I love USA and It's a place I want to study 🆚 I love USA and It's a place I want to study 
 Score 0.9999999999998701 

I love USA and It's a place I want to study 🆚 Let's write Python co

### Semantic Search

In [109]:
def semantic_search(query, documents, embedding_model, top_k=3):
    embeded_query = embedding_model.embed_query(query)
    embeded_documents = embedding_model.embed_documents(documents)
    similarity = []
    for i, emd_doc in enumerate(embeded_documents):
        similarity_score = cosine_similarity(embeded_query, emd_doc)
        similarity.append({"query": query, "document": documents[i], "score": similarity_score})
    similarity.sort(key=lambda x: x["score"], reverse=True)
    # top_k = max(1, top_k)
    if top_k < 1:
        raise ValueError("top_k must be at least 1")
    return (similarity[:top_k])

documents = [
    "Python is widely used for building modern software",
    "Love is a beautiful thing!",
    "Vector embeddings represent text as numbers",
    "Today's forecast shows sunshine and clear skies",
    "LangChain helps developers build LLM powered applications",
]

result = semantic_search("What is Love", documents, hf_embeddings, top_k=2)
df = pd.DataFrame(result)
df

,query,document,score
0,What is Love,Love is a beautiful thing!,0.650905
1,What is Love,Python is widely used for building modern soft...,0.079916
